# Network Expansion
# Model 1 - Deterministic Baseline

In [1]:
import numpy as np

from src.classes import DistributionNetwork
from src.classes import Substation
from src.solver import solve_network
from src.solver import print_results

![Topology example](figures/Model1_Initial.png)  
*Example topology of a distribution system evaluated in this notebook.*

#### Defining the Distribution Network: Input data

In [2]:
NODES = [f"N{i}" for i in range(1,14)] # Listing system nodes: 'N1','N2', ..., 'N13'
LOADS = [f'D{i}' for i in range(1,11)] # Listing system loads: 'D1', 'D2', ..., 'D10'
# NOTE: Candidate nodes (ones with potential substations) are added separately using add_candidate_substations method.

# Initial substations
S1 = Substation("S1", "N4", 40, ["N3", "N5", "N9"]) # Substation S1 at node N4, with capacity 40, connected to nodes N3, N5, N9
SUBSTATIONS = [S1] # Substations to pass to a class

line_cost = 50 # Default cost of building network lines (can be different from substation feeder lines)

load_capacity = {'D1': 5,
                 'D2': 2,
                 'D3': 2,
                 'D4': 5,
                 'D5': 3,
                 'D6': 2,
                 'D7': 6,
                 'D8': 5,
                 'D9': 3,
                 'D10': 4}

Mapping distribution lines and system loads (nodal locations)

In [3]:
# Loads
loads_locations = {
    "D1": "N1",
    "D2": "N2",
    "D3": "N3",
    "D4": "N6",
    "D5": "N7",
    "D6": "N8",
    "D7": "N9",
    "D8": "N11",
    "D9": "N12",
    "D10": "N13",
}

# Distribution lines
nodes_connected = {
    "N1": ["N2"],
    "N2": ["N1", "N3"],
    "N3": ["N2", "N4"],
    "N4": ["N3", "N5", "N9"],
    "N5": ["N4", "N6"],
    "N6": ["N5","N7", "N8"],
    "N7": ["N6"],
    "N8": ["N6"],
    "N9": ["N4", "N10"],
    "N10": ["N9", "N11", "N13"],
    "N11": ["N10", "N12"],
    "N12": ["N11"],
    "N13": ["N10"]
}

Instancing a pre-defined Distribution Network

In [4]:
DistributionNetwork = DistributionNetwork(NODES, 
                                          LOADS, 
                                          SUBSTATIONS, 
                                          load_capacity,
                                          nodes_connected,
                                          loads_locations,
                                          line_cost)

Adding *candidate* substations for expansion (with potential connection lines).

In [5]:
capacity = 20
s_cost = 100    # Cost of substation activation
l_cost = 50     # Cost of connecting a substation feeder line
S2 = Substation("S2", "N14", capacity, ["N2"], fix_cost = s_cost, edge_cost = l_cost) # Potential substation S2 at node N14, with potential connection to node N2
S3 = Substation("S3", "N15", capacity, ["N5"], fix_cost = s_cost, edge_cost = l_cost)
S4 = Substation("S4", "N16", capacity, ["N11", "N13"], fix_cost = s_cost, edge_cost = l_cost)
DistributionNetwork.add_candidate_substations([S2, S3, S4])

### Defining a Network Expansion optimization problem
Based on a radial operation of distribution systems, we modelled a baseline network expansion optimization problem. Network expansion here focuses on reinforcements in substations and feeder lines. Concretely, network can be reinforced in three ways:
1. Activating and connecting **new substations** to supply power in areas of high demand
2. Connecting new **feeder lines** (reconfiguring network) for power flow optimization
3. Reinforcing existing substations (adding capacity)

Both actions 1. and 2. can cause major network reconfiguration, as distribution networks operate radially, meaning each demand can be supplied by one, and only one, substation.  

Full `solve_network` function can be found in `src/solver.py`

1. **Parameters**

- $L_d:$ Power consumptiom of load $d$.
- $P_s:$ Capacity of substation $s$.
- $R:$ Size of a single capacity reinforcement.
- $C^{S}_s:$ Cost of activating substation $s$.
- $C^{L}_{(i,j)}:$ Cost of connecting line $(i,j)$.
- $C^R:$ Cost of capacity reinforcement.
- $B:$ Annual budget.
- $dr:$ Discount rate.
- $Y:$ Year (used only in Model 1 because of a singe-period optimization)
- $M:$ Big-M used to constraint $f$ (power flow). Equal to total system demand.
2. **Decision variables**
- $w_s \in \{0,1\}$ Substation Activation Variable. Indicates whether substation $s$ is activated.
- $y_{i,s} \in \{0,1\}$ Node Assignment Variable. Indicates whether node $i$ is served by substation $s$.
- $x_{i,j,s} \in \{0,1\}$ Arc Usage Variable. Indicates whether arc $(i,j)$ is used by flows from substation $s$. 
- $f_{s,i,j} \geq 0 $ Power Flow Variable. Represents the amount of power flow from substation $s$ travelling along directed arc $(i,j)$.
- $r_s \geq 0 $ Substation Supply. Represents total demand supplied by substation $s$.
- $z_s \in \Z$ Capacity Reinforcement Variable. Indicates how many times substation $s$ upgraded its maximum capacity.
3. **Constraints**  
- (1) - Power Balance
$$ ... $$
Description
- (2) - Non-substation Node Assignment
- (3) - Substation Node Assignment
- (4) - Initial Constraints
- (5) - Substation Supply
- (6) - Capacity Constraint
- (7) - Flow conservation
- (8) - Flow only if arc is assigned
- (9) - Distribution Network Radiality
- (10) - Tree size
- (11) - Annual Budget
4. **Model. Objective function**  
$$\min substation\ cost + feeder\ cost + capacity\ cost$$
where: 
- $substation\ cost = \sum\limits_{s=1}^{S} C^S_s w_s$ is a total cost of activating new substations
- $feeder\ cost = \sum\limits_{(i,j)\in A}^{}C^L_{(i,j)} x_{(i,j,s)}$ is a total cost of connecting new feeder lines
- $capacity\ cost = \sum\limits_{s=1}^{S} C^R r_s$ is a total cost of increasing capacity in existing substations

For simplification purposes, the model does not take into account the capacity or susceptance of distribution lines.

#### Solving the optimization problem

In [6]:
B = 300
dr = 0.05
Y = 1

results = {} # Storing yearly results

solution = solve_network(DistributionNetwork, B, dr, Y, OutputFlag=1)
DistributionNetwork.update_initial_conditions(solution['w'], solution['x'], solution['z'])
results[f"Y{Y}"] = solution

Set parameter Username
Set parameter LicenseID to value 2706854
Academic license - for non-commercial use only - expires 2026-09-10
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-9300H CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 335 rows, 332 columns and 1149 nonzeros
Model fingerprint: 0x756b1197
Model has 4 quadratic constraints
Variable types: 132 continuous, 200 integer (196 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  QMatrix range    [1e+01, 1e+01]
  QLMatrix range   [1e+00, 4e+01]
  Objective range  [5e+01, 2e+02]
  Bounds range     [1e+00, 4e+01]
  RHS range        [1e+00, 3e+02]
Presolve removed 279 rows and 278 columns
Presolve time: 0.02s
Presolved: 56 rows, 54 columns, 189 nonzeros
Variable types: 20 continuous, 34 integer (32 binary)
Found heuristi

Printing results

In [7]:
print_results(DistributionNetwork, solution)


Total cost: 0.0

Substation activation and supply:
S1 at N4: w=1.0, r=37.0
S2 at N14: w=0.0, r=0.0
S3 at N15: w=0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Node assignments:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1

Arcs used:
Arc N2 -> N1 assigned to S1
Arc N3 -> N2 assigned to S1
Arc N4 -> N3 assigned to S1
Arc N4 -> N5 assigned to S1
Arc N4 -> N9 assigned to S1
Arc N5 -> N6 assigned to S1
Arc N6 -> N7 assigned to S1
Arc N6 -> N8 assigned to S1
Arc N9 -> N10 assigned to S1
Arc N10 -> N11 assigned to S1
Arc N10 -> N13 assigned to S1
Arc N11 -> N12 assigned to S1


Solving the optimization problem for next years.  

Previous year's results are used as a new *DistributionNetwork* input.  
For Model 1 it is assumed that EV and heat-pump deployment cause an increase in demand at a constant rate - **increasing demand** in each load node uniformly **by 5.5%** annualy. The model is simplified by treating this demand as a peak demand, which is supplied at all times. This ultimately satisfies the 99% load-reliability constraint.

In [8]:
# Increasing demand
demand_rate = 0.055 # = 5.5% (can be also replaced by a linear increase)
for load, demand in DistributionNetwork.load_capacity.items():
    DistributionNetwork.load_capacity[load] = (1 + demand_rate) * demand

In [9]:
# Solving the problem
solution_2 = solve_network(DistributionNetwork, B, dr, 2)

# Updating initial substation activation constraints for the next time period
DistributionNetwork.update_initial_conditions(solution_2['w'], solution_2['x'], solution_2['z']) # this can be moved inside the solve_network function (tbd)

results[f"Y{2}"] = solution_2

print("Year 2")
print_results(DistributionNetwork, solution_2)

Year 2

Total cost: 0.0

Substation activation and supply:
S1 at N4: w=1.0, r=39.035
S2 at N14: w=0.0, r=0.0
S3 at N15: w=0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Node assignments:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1

Arcs used:
Arc N2 -> N1 assigned to S1
Arc N3 -> N2 assigned to S1
Arc N4 -> N3 assigned to S1
Arc N4 -> N5 assigned to S1
Arc N4 -> N9 assigned to S1
Arc N5 -> N6 assigned to S1
Arc N6 -> N7 assigned to S1
Arc N6 -> N8 assigned to S1
Arc N9 -> N10 assigned to S1
Arc N10 -> N11 assigned to S1
Arc N10 -> N13 assigned to S1
Arc N11 -> N12 assigned to S1


In [10]:
# Years 3-10
S = list(np.arange(1, len(DistributionNetwork.SUBSTATIONS)+1))  # [1,2, ...]
N = list(np.arange(1, len(DistributionNetwork.NODES)+1))        # [1,2, ...]

for y in range(3,11,1):
    for load, demand in DistributionNetwork.load_capacity.items():
        DistributionNetwork.load_capacity[load] = (1 + demand_rate) * demand    # Increase demand
    sol = solve_network(DistributionNetwork, B, dr, y)                          # Solve
    
    results[f"Y{y}"] = sol                                                      # Store results

    print(f"\nYear {y}")
    print("Total discounted cost:", round(sol['objective'],2))
    print("Substation activation and supply:")
    for s in S:
        print(f"{DistributionNetwork.SUBSTATIONS[s-1].id} at {DistributionNetwork.SUBSTATIONS[s-1].node}: w={sol['w'][s]}, r={sol['r'][s]:.2f}")

    DistributionNetwork.update_initial_conditions(sol['w'], sol['x'], sol['z']) # Update initial conditions for next time period


Year 3
Total discounted cost: 136.05
Substation activation and supply:
S1 at N4: w=1.0, r=31.16
S2 at N14: w=1.0, r=10.02
S3 at N15: w=0.0, r=0.00
S4 at N16: w=0.0, r=0.00
Arc disconnected: (3, 4)

Year 4
Total discounted cost: 0.0
Substation activation and supply:
S1 at N4: w=1.0, r=32.88
S2 at N14: w=1.0, r=10.57
S3 at N15: w=0.0, r=0.00
S4 at N16: w=0.0, r=0.00

Year 5
Total discounted cost: 0.0
Substation activation and supply:
S1 at N4: w=1.0, r=34.69
S2 at N14: w=1.0, r=11.15
S3 at N15: w=0.0, r=0.00
S4 at N16: w=0.0, r=0.00

Year 6
Total discounted cost: 0.0
Substation activation and supply:
S1 at N4: w=1.0, r=36.59
S2 at N14: w=1.0, r=11.76
S3 at N15: w=0.0, r=0.00
S4 at N16: w=0.0, r=0.00

Year 7
Total discounted cost: 0.0
Substation activation and supply:
S1 at N4: w=1.0, r=38.61
S2 at N14: w=1.0, r=12.41
S3 at N15: w=0.0, r=0.00
S4 at N16: w=0.0, r=0.00

Year 8
Total discounted cost: 106.6
Substation activation and supply:
S1 at N4: w=1.0, r=29.09
S2 at N14: w=1.0, r=13.09


In [11]:
print("\nNode assignments in Year 10:")
for i in N:
    for s in S:
        if results["Y10"]['y'][i,s] > 0.5:
            print(f"Node {DistributionNetwork.NODES[i-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Node assignments in Year 10:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S2
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S4
Node N12 assigned to S4
Node N13 assigned to S1
Node N14 assigned to S2
Node N16 assigned to S4


Results, plots, short insights

In [12]:
# TODO:

![Final example](figures/Model1_Final.png)  
*Illustration of Year 10 optimization results.*